# Brain Tumor Detection using YOLOv8
Training notebook for YOLOv8 on the Ultralytics brain-tumor dataset.

This notebook mirrors `train.ipynb` (Faster R-CNN) but uses YOLOv8.
mAP@0.5 target: **≥0.70** (baseline: 0.69 from Faster R-CNN)

## 1. Install Dependencies

In [ ]:
# Install ultralytics (includes YOLOv8)
!pip install -q ultralytics

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import pandas as pd
import yaml
import glob
import random
from PIL import Image
from pathlib import Path

from ultralytics import YOLO

sns.set(style='darkgrid')
print(f"YOLO version: {YOLO.__module__}")

## 2. Download Dataset

In [ ]:
import kagglehub

# Download latest version of Ultralytics brain-tumor dataset
path = kagglehub.dataset_download("ultralytics/brain-tumor")
print("Path to dataset files:", path)

In [ ]:
# Check the dataset structure
!ls "{path}/brain-tumor/train/images" | head -5
!echo "---"
!ls "{path}/brain-tumor/train/labels" | head -5
!echo "---"
!wc -l "{path}/brain-tumor/train/labels/"*.txt
!echo "---"
!cat "{path}/brain-tumor.yaml"

## 3. Dataset Configuration

YOLOv8 uses a YAML config file pointing to train/val image directories. The Ultralytics dataset already has this structure.

In [ ]:
yaml_path = os.path.join(path, "brain-tumor.yaml")
with open(yaml_path, "r") as f:
    bt_cfg = yaml.safe_load(f)

print("Dataset config:")
for k, v in bt_cfg.items():
    print(f"  {k}: {v}")

DATA_ROOT = os.path.join(path, "brain-tumor")
NUM_CLASSES = len(bt_cfg["names"])
CLASS_NAMES = [bt_cfg["names"][i] for i in sorted(bt_cfg["names"].keys())]
print(f"\nClasses ({NUM_CLASSES}): {CLASS_NAMES}")

## 4. Visualize Sample Images with YOLO Labels

In [ ]:
def plot_yolo_samples(image_dir, label_dir, num_samples=4):
    """Display images with their YOLO-format bounding boxes."""
    img_paths = sorted(glob.glob(os.path.join(image_dir, "*.jpg")))
    samples = random.sample(img_paths, min(num_samples, len(img_paths)))

    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    axes = axes.flatten()

    for ax, img_path in zip(axes, samples):
        im = Image.open(img_path).convert("RGB")
        w, h = im.size
        ax.imshow(im)
        ax.axis("off")

        # Load YOLO label
        stem = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(label_dir, stem + ".txt")
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls_id = int(parts[0])
                        cx, cy, bw, bh = map(float, parts[1:])
                        x1 = (cx - bw / 2) * w
                        y1 = (cy - bh / 2) * h
                        x2 = (cx + bw / 2) * w
                        y2 = (cy + bh / 2) * h
                        rect = patches.Rectangle(
                            (x1, y1), x2 - x1, y2 - y1,
                            linewidth=2, edgecolor="lime", facecolor="none"
                        )
                        ax.add_patch(rect)
                        ax.text(x1, y1 - 5, CLASS_NAMES[cls_id],
                                color="lime", fontsize=10,
                                bbox=dict(facecolor="black", alpha=0.5))

        ax.set_title(os.path.basename(img_path))

    plt.tight_layout()
    plt.show()

plot_yolo_samples(
    os.path.join(DATA_ROOT, "train", "images"),
    os.path.join(DATA_ROOT, "train", "labels"),
)

## 5. Train YOLOv8 Model

We train YOLOv8m (medium) for 50 epochs with early stopping.
YOLOv8 handles dataset splits, augmentation, and validation internally.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Where to save checkpoints and logs
CKPT_DIR = "/content/drive/MyDrive/yolov8_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
# ── Training configuration ───────────────────────────────────────

# Start from YOLOv8m pretrained on COCO
model = YOLO("yolov8m.pt")

# Training arguments — mirrors the Faster R-CNN training budget (10 epochs baseline)
# but YOLO benefits from more epochs, so we train longer with early stopping
results = model.train(
    data=yaml_path,          # dataset config
    epochs=50,               # max epochs (early stopping will cut shorter)
    patience=10,             # early stopping if no mAP improvement for 10 epochs
    batch=16,                # batch size
    imgsz=640,               # input image size
    optimizer="AdamW",       # optimizer
    lr0=0.001,               # initial learning rate
    lrf=0.01,                # final learning rate factor
    momentum=0.937,          # SGD momentum / Adam beta1
    weight_decay=0.0005,     # weight decay
    warmup_epochs=3,         # warmup epochs
    warmup_momentum=0.8,     # initial momentum during warmup
    warmup_bias_lr=0.1,      # learning rate for bias during warmup
    box=7.5,                 # box loss gain
    cls=0.5,                 # cls loss gain
    dfl=1.5,                 # dfl loss gain
    augment=True,            # use built-in augmentation
    val=True,               # validate after every epoch
    project=CKPT_DIR,
    name="yolov8m_brain_tumor",
    exist_ok=True,
    device="cuda" if os.path.exists("/usr/local/cuda") else "cpu",
    verbose=True,
)

### Training completed!

YOLOv8 automatically logs:
- Per-epoch metrics (precision, recall, mAP50, mAP50-95)
- Training losses (box_loss, cls_loss, dfl_loss)
- Learning rate schedule
- Best checkpoint saved as `best.pt`

The best model is at: `{CKPT_DIR}/yolov8m_brain_tumor/weights/best.pt`

## 6. Visualize Training Metrics

In [ ]:
results_dir = os.path.join(CKPT_DIR, "yolov8m_brain_tumor")

# Check if results.csv exists (YOLOv8 logs metrics here)
csv_path = os.path.join(results_dir, "results.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("Columns:", df.columns.tolist())
    display(df.tail(5))
else:
    print("results.csv not found yet — run training first")

In [ ]:
if os.path.exists(csv_path):
    metrics = [
        ("train/box_loss", "Box Loss"),
        ("train/cls_loss", "Class Loss"),
        ("train/dfl_loss", "DFL Loss"),
        ("metrics/precision(B)", "Precision"),
        ("metrics/recall(B)", "Recall"),
        ("metrics/mAP50(B)", "mAP@0.5"),
        ("metrics/mAP50-95(B)", "mAP@0.5-0.95"),
    ]

    fig, axes = plt.subplots(4, 2, figsize=(14, 14))
    axes = axes.flatten()

    for i, (col, title) in enumerate(metrics):
        if col in df.columns:
            axes[i].plot(df["epoch"], df[col], linewidth=2)
            axes[i].set_title(title, fontsize=13)
            axes[i].set_xlabel("Epoch")
            axes[i].grid(alpha=0.3)
            axes[i].set_xlim(1, df["epoch"].max())

    # Hide unused subplot (4x2=8, we have 7 metrics)
    axes[-1].set_visible(False)

    plt.suptitle("YOLOv8 Training Metrics", fontsize=18, y=1.02)
    plt.tight_layout()
    plt.show()

    # Print final metrics
    last = df.iloc[-1]
    print(f"\nFinal metrics (epoch {int(last['epoch'])}):")
    print(f"  mAP@0.5:      {last.get('metrics/mAP50(B)', 'N/A'):.4f}")
    print(f"  mAP@0.5-0.95: {last.get('metrics/mAP50-95(B)', 'N/A'):.4f}")
    print(f"  Precision:    {last.get('metrics/precision(B)', 'N/A'):.4f}")
    print(f"  Recall:       {last.get('metrics/recall(B)', 'N/A'):.4f}")

## 7. Evaluate on Test Set

In [ ]:
# Load the best model from training
best_ckpt = os.path.join(results_dir, "weights", "best.pt")
if os.path.exists(best_ckpt):
    model = YOLO(best_ckpt)
else:
    # Fall back to the final epoch checkpoint
    model = YOLO(os.path.join(results_dir, "weights", "last.pt"))

# Run validation on the test (valid) split
test_results = model.val(
    data=yaml_path,
    split="test",  # use the test/valid split
    conf=0.001,     # low threshold to capture all detections for mAP
    iou=0.5,        # NMS IoU threshold
    verbose=True,
)

print("\n================== TEST RESULTS ==================")
print(f"  mAP@0.5:      {test_results.box.map50:.4f}")
print(f"  mAP@0.5-0.95: {test_results.box.map:.4f}")
print(f"  Precision:    {test_results.box.mp:.4f}")
print(f"  Recall:       {test_results.box.mr:.4f}")
print("===================================================")

## 8. Visualize Predictions

In [ ]:
# Run inference on a few test images with overlay
test_img_dir = os.path.join(DATA_ROOT, "valid", "images")
test_img_paths = sorted(glob.glob(os.path.join(test_img_dir, "*.jpg")))
samples = random.sample(test_img_paths, min(8, len(test_img_paths)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, samples):
    # Run inference
    preds = model(img_path, conf=0.25, verbose=False)
    pred = preds[0]

    # Plot
    im = Image.open(img_path).convert("RGB")
    ax.imshow(im)
    ax.axis("off")

    if pred.boxes is not None:
        for box, conf, cls_id in zip(pred.boxes.xyxy, pred.boxes.conf, pred.boxes.cls):
            x1, y1, x2, y2 = box.tolist()
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor="red", facecolor="none"
            )
            ax.add_patch(rect)
            label = pred.names[int(cls_id)]
            ax.text(x1, y1 - 8, f"{label} {conf:.2f}",
                    color="yellow", fontsize=10,
                    bbox=dict(facecolor="black", alpha=0.6))

    ax.set_title(os.path.basename(img_path), fontsize=10)

plt.tight_layout()
plt.show()

## 9. Save Model for Deployment

Export the trained model in PyTorch format for the FastAPI backend.
The backend expects the checkpoint at `checkpoints/yolov8_tumor.pt`.

In [ ]:
# Export to plain PyTorch format (or keep .pt for YOLO)
export_path = os.path.join(CKPT_DIR, "yolov8_tumor.pt")
model.export(format="pt")  # already in pt format, this confirms the path

# Copy to a convenient location for download
!cp "{best_ckpt if os.path.exists(best_ckpt) else os.path.join(results_dir, 'weights', 'last.pt')}" "{export_path}"
print(f"\nModel saved to: {export_path}")

# Display file size
size_mb = os.path.getsize(export_path) / (1024 * 1024)
print(f"Model size: {size_mb:.1f} MB")

## 10. Compare with Faster R-CNN Baseline

| Model | mAP@0.5 | mAP@0.5-0.95 | Inference Speed |
|---|---|---|---|
| Faster R-CNN (ResNet-50) | 0.69 | 0.49 | ~200ms on T4 |
| YOLOv8m | **TBD** | **TBD** | **~15ms on T4** |

Fill in the YOLOv8 metrics from the evaluation cell above.
YOLOv8 typically achieves comparable accuracy at 10-15× the speed.